# Introducción a QA sobre noticias dominicanas

Recursos oficiales:

- Dataset: `Lisibonny/pdqa`
- Baseline: `Lisibonny/modelo_qa_beto_squad_es_pdqa`
- Space: `Lisibonny/Repartidor_Dominicano`

El dataset ya contiene las divisiones oficiales: `train`, `validation` y `test`.


In [1]:
!pip install -q "transformers>=4.45,<5.0" "datasets>=3.0,<5.0" "accelerate>=1.0,<2.0" "pandas>=2.0,<3.0" "pyarrow>=15,<22"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import random, string, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline

SEED = 42
DATASET_ID = "Lisibonny/pdqa"
BASELINE_MODEL_ID = "Lisibonny/modelo_qa_beto_squad_es_pdqa"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

dataset = load_dataset(DATASET_ID)
print(dataset)
assert set(["train","validation","test"]).issubset(dataset.keys())
print({split: len(dataset[split]) for split in dataset})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/8.68k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'context', 'answers', 'title'],
        num_rows: 45
    })
    validation: Dataset({
        features: ['id', 'question', 'context', 'answers', 'title'],
        num_rows: 15
    })
    test: Dataset({
        features: ['id', 'question', 'context', 'answers', 'title'],
        num_rows: 20
    })
})
{'train': 45, 'validation': 15, 'test': 20}


In [3]:
print(dataset["train"].column_names)
display(pd.DataFrame([dataset["train"][0]]))
display(pd.DataFrame([dataset["validation"][0]]))
display(pd.DataFrame([dataset["test"][0]]))


['id', 'question', 'context', 'answers', 'title']


,id,question,context,answers,title
0,5kpfplqtw2p99k1,¿Quiénes se van de “De Extremo a Extremo”?,Durante la pasada entrega de Premios Soberano ...,"{'answer_start': [171], 'text': ['Caroline Aqu...",prueba


,id,question,context,answers,title
0,c61s2xwyfi0cugi,¿Cuál es el objetivo de la campaña de Asindown?,Asindown ha lanzado una campaña de sensibiliza...,"{'answer_start': [161], 'text': ['concienciar ...",prueba


,id,question,context,answers,title
0,jygfnzhtybqz9u2,¿Quién escribió el poema “Una mujer está sola”?,"“Una mujer está sola”, escribió Aída Portalatí...","{'answer_start': [32], 'text': ['Aída Portalat...",prueba


## Verificación de columnas

El notebook espera columnas equivalentes a `id`, `question`, `context` y `answers`.
Si los nombres son distintos, ajuste solo las constantes siguientes.


In [4]:
ID_COL = "id"
QUESTION_COL = "question"
CONTEXT_COL = "context"
ANSWERS_COL = "answers"

required_train = {ID_COL, QUESTION_COL, CONTEXT_COL, ANSWERS_COL}
required_test = {ID_COL, QUESTION_COL, CONTEXT_COL}
assert required_train.issubset(dataset["train"].column_names)
assert required_train.issubset(dataset["validation"].column_names)
assert required_test.issubset(dataset["test"].column_names)


## Análisis exploratorio mínimo

In [5]:
train_df = dataset["train"].to_pandas()
validation_df = dataset["validation"].to_pandas()

for frame in [train_df, validation_df]:
    frame["question_words"] = frame[QUESTION_COL].astype(str).str.split().str.len()
    frame["context_words"] = frame[CONTEXT_COL].astype(str).str.split().str.len()

display(train_df[["question_words","context_words"]].describe())
display(validation_df[["question_words","context_words"]].describe())


,question_words,context_words
count,45.000000,45.000000
mean,6.244444,60.644444
std,2.612518,56.302412
min,3.000000,27.000000
25%,4.000000,34.000000
50%,6.000000,43.000000
75%,7.000000,55.000000
max,15.000000,285.000000


,question_words,context_words
count,15.000000,15.000000
mean,5.933333,98.133333
std,1.667619,112.981583
min,3.000000,29.000000
25%,5.000000,34.000000
50%,6.000000,38.000000
75%,6.000000,87.000000
max,9.000000,347.000000


## Inferencia con el baseline

In [6]:
device = 0 if torch.cuda.is_available() else -1
qa_baseline = pipeline(
    "question-answering",
    model=BASELINE_MODEL_ID,
    tokenizer=BASELINE_MODEL_ID,
    device=device,
)

example = dataset["validation"][0]
result = qa_baseline(
    question=example[QUESTION_COL],
    context=example[CONTEXT_COL],
)
print("Pregunta:", example[QUESTION_COL])
print("Referencia:", example[ANSWERS_COL])
print("Predicción:", result)


config.json:   0%|          | 0.00/712 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


Pregunta: ¿Cuál es el objetivo de la campaña de Asindown?
Referencia: {'answer_start': [161], 'text': ['concienciar sobre el acoso escolar y social que sufren las personas con síndrome de Down']}
Predicción: {'score': 0.09938068687915802, 'start': 161, 'end': 204, 'answer': 'concienciar sobre el acoso escolar y social'}


## Exact Match y F1

In [7]:
SPANISH_ARTICLES = {"el","la","los","las","un","una","unos","unas"}

def normalize_answer(text):
    text = "" if text is None else str(text).lower().strip()
    text = "".join(ch for ch in unicodedata.normalize("NFD", text)
                   if unicodedata.category(ch) != "Mn")
    punctuation = string.punctuation + "¡¿“”‘’«»…"
    text = "".join(" " if ch in punctuation else ch for ch in text)
    return " ".join(tok for tok in text.split() if tok not in SPANISH_ARTICLES)

def exact_match(pred, truth):
    return float(normalize_answer(pred) == normalize_answer(truth))

def token_f1(pred, truth):
    p = normalize_answer(pred).split()
    t = normalize_answer(truth).split()
    if not p and not t: return 1.0
    if not p or not t: return 0.0
    same = sum((Counter(p) & Counter(t)).values())
    if same == 0: return 0.0
    precision = same / len(p)
    recall = same / len(t)
    return 2 * precision * recall / (precision + recall)

def answer_texts(value):
    if isinstance(value, dict):
        value = value.get("text", value.get("answer", value))
    if isinstance(value, list):
        return [str(x) for x in value]
    return [str(value)]


In [8]:
rows = []
for ex in dataset["validation"]:
    pred = qa_baseline(question=ex[QUESTION_COL], context=ex[CONTEXT_COL])
    golds = answer_texts(ex[ANSWERS_COL])
    em = max(exact_match(pred["answer"], g) for g in golds)
    f1 = max(token_f1(pred["answer"], g) for g in golds)
    rows.append({
        "id": ex[ID_COL],
        "question": ex[QUESTION_COL],
        "reference": " | ".join(golds),
        "prediction": pred["answer"],
        "confidence": pred["score"],
        "exact_match": em,
        "f1": f1,
    })

baseline_results = pd.DataFrame(rows)
display(baseline_results)
print("Exact Match:", 100 * baseline_results["exact_match"].mean())
print("F1:", 100 * baseline_results["f1"].mean())


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,id,question,reference,prediction,confidence,exact_match,f1
0,c61s2xwyfi0cugi,¿Cuál es el objetivo de la campaña de Asindown?,concienciar sobre el acoso escolar y social qu...,concienciar sobre el acoso escolar y social,0.099381,0.0,0.631579
1,7857zba14eers9x,¿Cómo se llama el canal 4?,Radio Televisión Dominicana,Radio Televisión Dominicana,0.634700,1.0,1.000000
2,hkhjremdmrmix29,¿Cuál es nuestro símbolo patrio?,La Bandera Nacional,La Bandera Nacional,0.382562,1.0,1.000000
3,ayfu15q9bfn5nhb,¿Quién fundó Menudo?,Edgardo García,Edgardo García,0.545798,1.0,1.000000
4,rpcbxp9rc8g0iuh,¿Cuando es el Miércoles de Ceniza?,El próximo miércoles 22 de febrero,El próximo miércoles 22 de febrero,0.336815,1.0,1.000000
5,d2xt90fah10kox5,¿A cuántos bateadores ponchó Jacob deGrom?,a 11 bateadores,11 bateadores,0.367446,0.0,0.800000
6,sz9yeioprmn61xv,¿Quién es el creador de Dilbert?,Scott Adams,Scott Adams,0.959120,1.0,1.000000
7,ak5ubvypoj83wfp,¿Cuántos minutos jugó Immanuel Quickley?,55 minutos,55 minutos,0.447666,1.0,1.000000
8,3u77e1org81bc7f,¿Qué jugador estaba lesionado?,Jalen Brunson,Jalen Brunson,0.796943,1.0,1.000000
9,pxqbvfcey7z9p3q,¿Cómo será la inflación en 2023?,seguirá siendo alta,en torno al 7%,0.187205,0.0,0.000000


Exact Match: 60.0
F1: 75.84015594541911


##Evaluación del baseline sobre el split de validation

In [9]:
rows = []
for ex in dataset["validation"]:
    pred = qa_baseline(question=ex[QUESTION_COL], context=ex[CONTEXT_COL])
    golds = answer_texts(ex[ANSWERS_COL])
    em = max(exact_match(pred["answer"], g) for g in golds)
    f1 = max(token_f1(pred["answer"], g) for g in golds)
    rows.append({
        "id": ex[ID_COL],
        "question": ex[QUESTION_COL],
        "reference": " | ".join(golds),
        "prediction": pred["answer"],
        "confidence": pred["score"],
        "exact_match": em,
        "f1": f1,
    })

baseline_results = pd.DataFrame(rows)
baseline_results.to_csv("baseline_results.csv", index=False)

em_baseline = 100 * baseline_results["exact_match"].mean()
f1_baseline = 100 * baseline_results["f1"].mean()
print(f"Exact Match: {em_baseline:.2f}%")
print(f"F1: {f1_baseline:.2f}%")

Exact Match: 60.00%
F1: 75.84%


<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

</div>

##Análisis de errores

In [10]:
def categorizar_error(row):
    if row["exact_match"] == 1.0:
        return "Acierto exacto"
    if row["f1"] >= 0.5:
        return "Truncamiento de límites (respuesta parcialmente correcta)"
    return "Fallo total (respuesta equivocada o entidad incorrecta)"

baseline_results["categoria"] = baseline_results.apply(categorizar_error, axis=1)

resumen_categorias = baseline_results["categoria"].value_counts()
print(resumen_categorias)
print()

# Ejemplos de cada categoría de error
errores = baseline_results[baseline_results["exact_match"] == 0].copy()
errores["longitud_referencia"] = errores["reference"].str.split().str.len()
display(errores[["question", "reference", "prediction", "f1", "categoria", "longitud_referencia"]])

categoria
Acierto exacto                                               9
Truncamiento de límites (respuesta parcialmente correcta)    3
Fallo total (respuesta equivocada o entidad incorrecta)      3
Name: count, dtype: int64



,question,reference,prediction,f1,categoria,longitud_referencia
0,¿Cuál es el objetivo de la campaña de Asindown?,concienciar sobre el acoso escolar y social qu...,concienciar sobre el acoso escolar y social,0.631579,Truncamiento de límites (respuesta parcialment...,15
5,¿A cuántos bateadores ponchó Jacob deGrom?,a 11 bateadores,11 bateadores,0.800000,Truncamiento de límites (respuesta parcialment...,3
9,¿Cómo será la inflación en 2023?,seguirá siendo alta,en torno al 7%,0.000000,Fallo total (respuesta equivocada o entidad in...,3
10,¿De qué trata la película Ramona?,gira en torno al embarazo en adolescentes,torno al embarazo en adolescentes,0.833333,Truncamiento de límites (respuesta parcialment...,7
11,¿Porqué cerro al nivel más bajo el petróleo?,por las turbulencias en el sector bancario en ...,desde diciembre,0.000000,Fallo total (respuesta equivocada o entidad in...,12
13,¿A quiénes apresó la Procuraduría?,"los exministros: José Ramón Peralta, Administr...",Guerrero,0.111111,Fallo total (respuesta equivocada o entidad in...,19


##Errores del baseline

El modelo baseline obtiene **EM = 60%** y **F1 = 75.8%**. La brecha de ~16 puntos
entre ambas métricas es la primera señal importante: el modelo casi siempre
identifica la región correcta del contexto donde está la respuesta, pero no
siempre delimita el span exacto (de ahí que F1, que mide superposición de
palabras, sea consistentemente más alto que EM, que exige coincidencia perfecta).

Categorizando los 6 casos con exact_match = 0, aparecen dos patrones distintos:

**1. Truncamiento de límites (F1 ≥ 0.5, 3 de 6 casos)**
El modelo ubica la respuesta correctamente pero corta la frase de más o de menos:
- "objetivo de la campaña Asindown" → predijo la primera mitad de la oración,
  omitiendo la cláusula final ("...que sufren las personas con síndrome de Down").
- "bateadores ponchados por deGrom" → omitió el artículo inicial ("a 11" vs "11").
- "de qué trata Ramona" → cortó la primera palabra de la frase ("gira").

Estos casos tienen F1 alto (0.63–0.83) porque comparten casi todas las palabras
con la referencia; el error es de **frontera de extracción**, no de comprensión.
Es razonable hipotetizar que más entrenamiento (épocas) ayude al modelo a
calibrar mejor dónde empieza y termina el span, dado que el dataset de
entrenamiento es muy pequeño (45 ejemplos) para que esto se aprenda en pocos pasos.

**2. Fallo total (F1 < 0.2, 3 de 6 casos)**
El modelo extrae información del contexto que no corresponde a la pregunta:
- "inflación en 2023" → el contexto probablemente menciona varias cifras
  numéricas; el modelo extrajo un dato relacionado pero no la respuesta
  correcta (respuesta esperada: una descripción cualitativa, no un número).
- "por qué cerró bajo el petróleo" → respondió con una referencia temporal
  ("desde diciembre") en vez de la causa que pedía la pregunta.
- "a quiénes apresó la Procuraduría" → la referencia es una lista de tres
  personas con cargos; el modelo solo extrajo un nombre aislado.

Este segundo grupo no mejora con más entrenamiento del mismo tipo: son casos
donde el contexto tiene **múltiples entidades o cifras candidatas** y el modelo
no tiene forma de distinguir cuál es la que responde específicamente la
pregunta. Es una limitación más estructural (posiblemente de cómo el modelo
fue expuesto a preguntas de tipo "por qué" o respuestas compuestas/listas
durante el entrenamiento original), y es poco probable que se resuelva solo
ajustando learning rate o épocas — necesitaría más ejemplos de este tipo de
pregunta en el dataset de entrenamiento.

## Preparación del dataset: tokenización y localización del span de respuesta


In [11]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = BASELINE_MODEL_ID
tokenizer_ft = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

MAX_LENGTH = 384
DOC_STRIDE = 128

def preparar_features(ejemplos):
    preguntas = [q.strip() for q in ejemplos["question"]]
    tokenized = tokenizer_ft(
        preguntas,
        ejemplos["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = ejemplos["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = tokenized.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

train_tokenizado = dataset["train"].map(
    preparar_features, batched=True, remove_columns=dataset["train"].column_names
)
print(train_tokenizado)

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 47
})


##Función para entrenar variantes (fine-tuning controlado)

In [12]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

def entrenar_variante(nombre, learning_rate, epochs=3, batch_size=8, weight_decay=0.01):
    modelo = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)

    args = TrainingArguments(
        output_dir=f"./{nombre}",
        learning_rate=learning_rate,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        weight_decay=weight_decay,
        seed=42,
        save_strategy="epoch",
        logging_steps=5,
        report_to="none",
    )

    trainer = Trainer(
        model=modelo,
        args=args,
        train_dataset=train_tokenizado,
        tokenizer=tokenizer_ft,
    )

    trainer.train()
    trainer.save_model(f"./{nombre}_final")
    tokenizer_ft.save_pretrained(f"./{nombre}_final")
    return f"./{nombre}_final"

## Variante 1: aumento de épocas de entrenamiento

**Hiperparámetro modificado:** `num_train_epochs` (4 → 10). El resto de la
configuración se mantiene igual al baseline original: `learning_rate = 2e-05`,
`batch_size = 16`, `seed = 42`.

**Hipótesis:** en el análisis de errores del baseline, identificamos que 3 de
los 6 fallos correspondían a un patrón de "truncamiento de límites" — el modelo
ubicaba correctamente la región de la respuesta en el contexto, pero cortaba
el span de más o de menos (ej. "11 bateadores" en vez de "a 11 bateadores").
Este tipo de error tiene F1 alto pero Exact Match bajo, lo que sugiere que el
modelo entiende dónde está la respuesta, pero no ha aprendido a delimitar sus
bordes con precisión.

Dado que el dataset de entrenamiento es muy pequeño (45 ejemplos) y el baseline
original solo fue entrenado con 4 épocas, hipotetizamos que el modelo no tuvo
suficientes pasadas por los datos para afinar estos límites exactos. Aumentar
las épocas a 10 le da al modelo más oportunidades de ajustar sus pesos en
relación a este patrón específico, sin cambiar ningún otro factor del
entrenamiento — así aislamos el efecto de esta única variable.

**Riesgo a vigilar:** con un dataset tan chico, más épocas también aumenta el
riesgo de sobreajuste (overfitting). Por eso evaluamos el resultado sobre
`validation` (nunca sobre `train`) para verificar que la mejora sea real y no
memorización.

In [13]:
ruta_variante1 = entrenar_variante("variante1_epochs", learning_rate=2e-05, epochs=10, batch_size=16)
print("Modelo guardado en:", ruta_variante1)

/tmp/ipykernel_920/2943336754.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
5,0.748100
10,0.388600
15,0.173000
20,0.150000
25,0.119600
30,0.099500


Modelo guardado en: ./variante1_epochs_final


### Evaluación de la Variante 1 sobre `validation`

Una vez entrenada la Variante 1, cargamos el modelo guardado en `ruta_variante1` y lo evaluamos utilizando el conjunto de `validation`. Para que la comparación con el baseline sea justa, seguimos exactamente el mismo procedimiento de evaluación: generamos una predicción para cada una de las 15 preguntas y calculamos las métricas Exact Match (EM) y F1 comparándolas con las respuestas de referencia. De esta manera, cualquier diferencia en el rendimiento se debe únicamente al modelo y no al método de evaluación.


In [14]:
qa_variante1 = pipeline(
    "question-answering",
    model=ruta_variante1,
    tokenizer=ruta_variante1,
    device=device,
)

rows_v1 = []
for ex in dataset["validation"]:
    pred = qa_variante1(question=ex[QUESTION_COL], context=ex[CONTEXT_COL])
    golds = answer_texts(ex[ANSWERS_COL])
    em = max(exact_match(pred["answer"], g) for g in golds)
    f1 = max(token_f1(pred["answer"], g) for g in golds)
    rows_v1.append({"id": ex[ID_COL], "prediction": pred["answer"], "exact_match": em, "f1": f1})

variante1_results = pd.DataFrame(rows_v1)
variante1_results.to_csv("variante1_results.csv", index=False)
print("EM Variante 1:", 100 * variante1_results["exact_match"].mean())
print("F1 Variante 1:", 100 * variante1_results["f1"].mean())

Device set to use cuda:0


EM Variante 1: 66.66666666666666
F1 Variante 1: 77.69770580296897


### Resultado de la Variante 1

| Métrica | Baseline | Variante 1 | Diferencia |
|---|---|---|---|
| Exact Match | 60.00% | 66.67% | +6.67 |
| F1 | 75.84% | 77.70% | +1.86 |

**Análisis:** aumentar las épocas de 4 a 10 mejoró tanto EM como F1, aunque el
efecto fue puntual: de las 15 predicciones, solo 3 cambiaron respecto al
baseline, y únicamente 1 caso pasó de incorrecto a correcto en Exact Match
(el de "bateadores ponchados por deGrom", que antes omitía el artículo "a" al
inicio de la respuesta). Este caso corresponde exactamente al patrón de
"truncamiento de límites" identificado en el análisis de errores del
baseline, lo que confirma parcialmente la hipótesis: más épocas sí ayudan al
modelo a delimitar mejor los bordes de la respuesta en casos límite. Sin
embargo, los errores más graves (respuestas totalmente equivocadas, como
"inflación 2023" o "petróleo") no se modificaron en absoluto — esto sugiere
que ese segundo tipo de error no depende de la cantidad de entrenamiento, sino
de una limitación distinta (probablemente ambigüedad genuina del contexto o
falta de ejemplos similares en el dataset de entrenamiento).

**Conclusión:** la Variante 1 mejora de forma consistente sobre el baseline
sin empeorar ningún caso, por lo que queda como candidata sólida a modelo
final.

## Variante 2: aumento del weight decay

**Hiperparámetro modificado:** `weight_decay` (0.01 → 0.1). Todos los demás
parámetros se mantienen iguales al baseline: `learning_rate = 2e-05`,
`num_train_epochs = 4`, `batch_size = 16` y `seed = 42`.

**¿Por qué modificar el weight decay?** El weight decay es una técnica de
regularización que penaliza a los pesos del modelo durante el entrenamiento,
restándoles en cada actualización una fracción proporcional a su propio
valor, empujándolos a mantenerse pequeños. La idea detrás de esto es evitar
que el modelo memorice detalles muy específicos del conjunto de entrenamiento
(overfitting), favoreciendo en su lugar patrones más generales que
generalicen mejor a ejemplos nuevos.

En este caso, el conjunto de entrenamiento tiene solo 45 ejemplos, una
cantidad lo suficientemente pequeña como para que el riesgo de sobreajuste
sea real: el modelo podría estar ajustándose demasiado a las particularidades
de esos ejemplos puntuales en vez de aprender el patrón general de cómo
delimitar una respuesta dentro de un contexto.

**Hipótesis:** al aumentar el `weight_decay` a 0.1, esperamos que el modelo
se vea forzado a aprender representaciones más generales en lugar de
memorizar los ejemplos vistos. A diferencia de la Variante 1, donde se buscó
darle al modelo más oportunidades de ajuste (más épocas), en esta variante no
se modifica la cantidad de entrenamiento, sino la forma en que el modelo es
penalizado durante ese entrenamiento — si el problema real fuera sobreajuste,
esta variable debería mostrar una mejora en `validation` sin necesidad de
más épocas ni más pasos de actualización.

**Aspecto a tener en cuenta:** un weight_decay demasiado alto puede tener el
efecto contrario al deseado: si penaliza los pesos con demasiada fuerza, el
modelo puede no lograr ajustarse lo suficiente ni siquiera a los patrones
útiles del dataset (underfitting). Por ello, el desempeño se evalúa sobre el
conjunto de `validation` y no sobre `train`, para comprobar si el aumento del
weight_decay realmente mejora la capacidad de generalización del modelo, o si
por el contrario perjudica su desempeño.

In [20]:
ruta_variante2 = entrenar_variante(
    "variante2_weightdecay",
    learning_rate=2e-05,
    epochs=4,
    batch_size=16,
    weight_decay=0.1,
)
print("Modelo guardado en:", ruta_variante2)

/tmp/ipykernel_920/2943336754.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
5,0.754000
10,0.402500


Modelo guardado en: ./variante2_weightdecay_final


##Evaluación de la Variante 2 sobre validation


Una vez entrenada la Variante 2, cargamos el modelo guardado en `ruta_variante2`
y lo evaluamos utilizando el conjunto de `validation`. Para que la comparación
con el baseline y la Variante 1 sea justa, seguimos exactamente el mismo
procedimiento de evaluación: generamos una predicción para cada una de las 15
preguntas y calculamos las métricas Exact Match (EM) y F1 comparándolas con
las respuestas de referencia. De esta manera, cualquier diferencia en el
rendimiento se debe únicamente al cambio en `weight_decay` y no al método de
evaluación.

In [21]:
qa_variante2 = pipeline(
    "question-answering",
    model=ruta_variante2,
    tokenizer=ruta_variante2,
    device=device,
)

rows_v2 = []
for ex in dataset["validation"]:
    pred = qa_variante2(question=ex[QUESTION_COL], context=ex[CONTEXT_COL])
    golds = answer_texts(ex[ANSWERS_COL])
    em = max(exact_match(pred["answer"], g) for g in golds)
    f1 = max(token_f1(pred["answer"], g) for g in golds)
    rows_v2.append({"id": ex[ID_COL], "prediction": pred["answer"], "exact_match": em, "f1": f1})

variante2_results = pd.DataFrame(rows_v2)
variante2_results.to_csv("variante2_results.csv", index=False)
print("EM Variante 2:", 100 * variante2_results["exact_match"].mean())
print("F1 Variante 2:", 100 * variante2_results["f1"].mean())

Device set to use cuda:0


EM Variante 2: 66.66666666666666
F1 Variante 2: 77.69770580296897


### Resultado de la Variante 2

| Métrica | Baseline | Variante 1 | Variante 2 | Diferencia vs baseline |
|---|---|---|---|---|
| Exact Match | 60.00% | 66.67% | 66.67% | +6.67 |
| F1 | 75.84% | 77.70% | 77.70% | +1.86 |

**Análisis:** aumentar el weight_decay de 0.01 a 0.1 también mejoró el
desempeño respecto al baseline, alcanzando exactamente el mismo resultado que
la Variante 1 (más épocas). Al comparar las predicciones fila por fila, ambas
variantes corrigieron el mismo caso puntual del baseline (el patrón de
"truncamiento de límites" identificado en el análisis de errores), y
mantuvieron el resto de las predicciones sin cambios. Esto es un hallazgo
interesante: dos intervenciones conceptualmente distintas —una que regula el
crecimiento de los pesos (weight decay) y otra que aumenta la cantidad de
pasadas por los datos (épocas)— convergieron al mismo punto de mejora. Esto
sugiere que, con un dataset de entrenamiento tan pequeño (45 ejemplos), existe
un techo de mejora alcanzable por distintos caminos, mientras que los errores
más graves del baseline (fallos totales por ambigüedad del contexto, como las
preguntas sobre inflación o el precio del petróleo) no se modifican con
ningún ajuste de hiperparámetros probado.

**Conclusión:** la Variante 2 iguala a la Variante 1 en ambas métricas, por lo
que ambas quedan como candidatas válidas a modelo final. Se elige la Variante
1 por simplicidad (modifica un solo hiperparámetro directamente relacionado
con la cantidad de entrenamiento, sin necesitar regularización adicional),
aunque el resultado de la Variante 2 respalda la misma conclusión.

## Selección del modelo final
Comparando las configuraciones evaluadas sobre `validation`:

| Modelo | Learning rate | Epochs | Batch size | Weight decay | EM | F1 |
|---|---|---|---|---|---|---|
| Baseline | 2e-05 | 4 | 16 | 0.01 | 60.00% | 75.84% |
| Variante 1 (épocas) | 2e-05 | 10 | 16 | 0.01 | **66.67%** | **77.70%** |
| Variante 2 (weight decay) | 2e-05 | 4 | 16 | 0.1 | **66.67%** | **77.70%** |

**Modelo final elegido: Variante 1 (aumento de épocas a 10).**

**Justificación:** ambas variantes evaluadas mejoraron sobre el baseline en
igual magnitud (EM +6.67, F1 +1.86), corrigiendo el mismo caso específico de
"truncamiento de límites". Se elige la Variante 1 como modelo final por ser la
configuración más simple de justificar y reproducir: modifica un único
hiperparámetro directamente vinculado a la cantidad de entrenamiento
(épocas), sin depender de un mecanismo de regularización adicional. La
Variante 2 respalda la misma conclusión desde un ángulo distinto, lo que
refuerza la confianza en el resultado.

**Limitación que persiste:** ninguna de las dos variantes logró corregir los
3 errores de "fallo total" identificados en el baseline (preguntas sobre
inflación, precio del petróleo, y la lista de exministros apresados). Estos
casos comparten un patrón distinto: el modelo extrae información del
contexto que no corresponde a lo que pregunta específicamente, sugiriendo que
el problema no es de cantidad de entrenamiento ni de regularización, sino de
ambigüedad genuina frente a contextos con múltiples entidades o cifras
candidatas. Resolver esto probablemente requeriría más ejemplos de
entrenamiento con ese patrón específico, no solo ajustes de hiperparámetros.


In [17]:
FINAL_MODEL_ID = ruta_variante1
qa_final = pipeline(
    "question-answering",
    model=FINAL_MODEL_ID,
    tokenizer=FINAL_MODEL_ID,
    device=device,
)

submission_rows = []
for ex in dataset["test"]:
    pred = qa_final(question=ex[QUESTION_COL], context=ex[CONTEXT_COL])
    submission_rows.append({"id": ex[ID_COL], "prediction": pred["answer"]})

submission = pd.DataFrame(submission_rows)
submission.to_csv("submission.csv", index=False, encoding="utf-8-sig")
display(submission)

Device set to use cuda:0


,id,prediction
0,jygfnzhtybqz9u2,Aída Portalatín
1,9zptnl85o2ocses,abraza la violencia a la mujer
2,gm0qmv6gv0qh1hu,Partido de la Liberación Dominicana
3,m9fzivkm3yoslwg,4 diputados
4,elvla3nmkr4u3rk,frente al Palacio de Justicia de Ciudad Nueva
5,4lp1ayfq8cavvo8,Royce
6,jct9x3le3n2oc7m,por la divulgación de contenido para adultos e...
7,7wfhp30e28oxep5,orientadora de los estudiantes del liceo Franc...
8,xnk6ge45wkyul3m,artista urbano puertorriqueño
9,j0wjntn0ot11fpr,en el 112 Dyckman Street en Manhattan


In [25]:
!python 07_validate_submission.py submission.csv

Filas leídas: 20
Resultado: ENTREGA VÁLIDA


In [26]:
from huggingface_hub import notebook_login
notebook_login()

In [27]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

modelo_final = AutoModelForQuestionAnswering.from_pretrained(ruta_variante1)
tokenizer_final = AutoTokenizer.from_pretrained(ruta_variante1)

NOMBRE_REPO = "Majo17/modelo_qa_beto_pdqa_variante1"

modelo_final.push_to_hub(NOMBRE_REPO)
tokenizer_final.push_to_hub(NOMBRE_REPO)

print(f"Modelo subido en: https://huggingface.co/{NOMBRE_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...59cz93c/model.safetensors:   0%|          |  554kB /  437MB            

README.md: 0.00B [00:00, ?B/s]

Modelo subido en: https://huggingface.co/Majo17/modelo_qa_beto_pdqa_variante1


## Lista de comprobación

- [✅ ] Evalué el baseline en validation.
- [✅ ] Comparé al menos dos variantes.
- [✅] Registré hiperparámetros y semillas.
- [✅] Realicé una ablación o comparación controlada.
- [✅] Analicé errores.
- [✅] Generé y validé submission.csv.
- [✅] Publiqué el modelo y completé la model card.
